In [1]:
import numpy as np
import pickle
import cv2
from mtcnn import MTCNN
from PIL import Image
from numpy import asarray
from numpy import expand_dims
from keras_facenet import FaceNet

In [5]:
# Load database embedding
try:
    with open("model.pkl", "rb") as myfile:
        database = pickle.load(myfile)
except Exception as e:
    print(f"Error loading database: {e}")
    exit()

In [13]:
def rescale_frame(frame, scale=0.5):
    """
    Fungsi untuk merubah ukuran frame video.
    :param frame: Frame video input
    :param scale: Skala pengurangan ukuran (default 0.5)
    :return: Frame video yang telah diskalakan
    """
    width = int(frame.shape[1] * scale)
    height = int(frame.shape[0] * scale)
    dimensions = (width, height)
    return cv2.resize(frame, dimensions, interpolation=cv2.INTER_AREA)

THRESHOLD = 1
detector = MTCNN()
MyFaceNet = FaceNet()

# Muat database wajah
with open("model.pkl", "rb") as myfile:
    database = pickle.load(myfile)

video_path = "testing/mas zuckerberg.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"Error: Tidak dapat membuka file video {video_path}.")
    exit()

print("Tekan 'Enter' untuk keluar.")

while True:
    try:
        # Baca frame dari video
        ret, frame = cap.read()
        if not ret:
            print("Video selesai diputar.")
            break

        # Deteksi wajah menggunakan MTCNN
        faces = detector.detect_faces(frame)

        if len(faces) > 0:
            for face in faces:
                x1, y1, width, height = face['box']
                x2, y2 = x1 + width, y1 + height

                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                cropped_face = rgb_frame[y1:y2, x1:x2]

                if cropped_face.shape[0] == 0 or cropped_face.shape[1] == 0:
                    print("Wajah tidak valid, dilewati.")
                    continue

                face = Image.fromarray(cropped_face).resize((160, 160))
                face = asarray(face)

                # Ekspansi dimensi untuk prediksi model
                face = expand_dims(face, axis=0)
                signature = MyFaceNet.embeddings(face)

                min_dist = float("inf")
                identity = "Unknown"  # Default label untuk wajah yang tidak dikenali
                nim = ""  # Default kosong untuk NIM

                # Bandingkan dengan database
                for key, value in database.items():
                    # Ambil embedding dari value yang merupakan dictionary
                    dist = np.linalg.norm(value['embedding'] - signature)  # Akses embedding dengan key 'embedding'
                    if dist < min_dist:
                        min_dist = dist
                        if dist < THRESHOLD:
                            identity = value['name']  # Mengambil nama jika wajah dikenali
                            nim = value['nim']  # Mengambil NIM
                        else:
                            identity = "Unknown"
                            nim = ""

                # Menampilkan nama dan NIM di frame
                cv2.putText(frame, f"{identity} ({nim})", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Resize frame untuk ditampilkan lebih kecil
        scaled_frame = rescale_frame(frame, scale=0.5)

        # Tampilkan frame
        cv2.imshow('Face Recognition', scaled_frame)

        # Keluar dengan menekan tombol 'Enter'
        if cv2.waitKey(1) & 0xFF == 13:
            break

    except Exception as e:
        print(f"Error: {e}")
        break

cap.release()
cv2.destroyAllWindows()

Tekan 'Enter' untuk keluar.
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━

In [8]:
# # Load HaarCascade untuk deteksi wajah
# HaarCascade = cv2.CascadeClassifier(cv2.samples.findFile(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'))
#
# MyFaceNet = FaceNet()

In [9]:
# THRESHOLD = 1
# detector = MTCNN()
# MyFaceNet = FaceNet()
#
# myfile = open("model.pkl", "rb")
# database = pickle.load(myfile)
# myfile.close()
#
# cap = cv2.VideoCapture(0)
#
# if not cap.isOpened():
#     print("Error: Tidak dapat mengakses kamera.")
#     exit()
#
# print("Tekan 'Enter' untuk keluar.")
#
# while True:
#     try:
#         # Baca frame dari kamera
#         ret, frame = cap.read()
#         if not ret:
#             print("Error: Tidak dapat membaca frame dari kamera.")
#             break
#
#         # Deteksi wajah menggunakan MTCNN
#         faces = detector.detect_faces(frame)
#
#         # Jika ada wajah yang terdeteksi
#         if len(faces) > 0:
#             for face in faces:
#                 x1, y1, width, height = face['box']
#                 x2, y2 = x1 + width, y1 + height
#
#                 # Potong wajah dan lakukan pemrosesan lebih lanjut
#                 rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#                 cropped_face = rgb_frame[y1:y2, x1:x2]
#
#                 # Validasi dimensi wajah
#                 if cropped_face.shape[0] == 0 or cropped_face.shape[1] == 0:
#                     print("Wajah tidak valid, dilewati.")
#                     continue
#
#                 # Resize wajah ke 160x160
#                 face = Image.fromarray(cropped_face).resize((160, 160))
#                 face = asarray(face)
#
#                 # Ekspansi dimensi untuk prediksi model
#                 face = expand_dims(face, axis=0)
#                 signature = MyFaceNet.embeddings(face)
#
#                 # Identifikasi wajah
#                 min_dist = float("inf")
#                 identity = "Unknown"  # Default label untuk wajah yang tidak dikenali
#
#                 # Bandingkan dengan database
#                 for key, value in database.items():
#                     dist = np.linalg.norm(value - signature)
#                     if dist < min_dist:
#                         min_dist = dist
#                         identity = key if dist < THRESHOLD else "Unknown"
#
#                 # Tambahkan label dan kotak ke frame
#                 cv2.putText(frame, identity, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
#                 cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
#
#         # Tampilkan frame
#         cv2.imshow('Face Recognition', frame)
#
#         # Keluar jika 'Enter' ditekan
#         if cv2.waitKey(1) & 0xFF == 13:  # Tekan Enter untuk keluar
#             break
#
#     except Exception as e:
#         print(f"Error: {e}")
#         break
#
# cap.release()
# cv2.destroyAllWindows()